# r2-001 Problem 3 — Solution (programming Parts 3.5–3.8)

Reference solution for Parts 3.5–3.8 of `problems/p03.ipynb`.
Theory parts 3.1–3.4 are in `answers.md`.
The cell below is the problem notebook's supplied setup cell, unchanged.

In [ ]:
import importlib.util
import json
import math
import os
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261101
torch.set_num_threads(1)


def find_data_dir() -> Path:
    """Locate r2-001's data/ folder from the notebook's working directory."""
    candidates = [Path("../data"), Path("data"), Path("mocktests/r2-001/data"), Path("book2/mocktests/r2-001/data")]
    env_root = os.environ.get("USAAIO_BOOK_ROOT")
    if env_root:
        candidates.insert(0, Path(env_root) / "mocktests" / "r2-001" / "data")
    for path in candidates:
        if (path / "gen_r2_001.py").is_file():
            return path.resolve()
    raise FileNotFoundError("cannot find mocktests/r2-001/data; run from problems/ or set USAAIO_BOOK_ROOT")


DATA_DIR = find_data_dir()

vae_rows = json.loads((DATA_DIR / "p03_vae.json").read_text(encoding="utf-8"))
x_train = torch.tensor(vae_rows["train"], dtype=torch.float32)
x_heldout = torch.tensor(vae_rows["heldout"], dtype=torch.float32)
BETA = 2.0
print(x_train.shape, x_heldout.shape)

## Part 3.5 — KL to the prior and the reparameterization

In [ ]:
def kl_to_standard_normal(mu, logvar):
    """Per-example KL(N(mu, diag e^logvar) || N(0, I)), summed over latents: shape (B,)."""
    return 0.5 * (mu ** 2 + torch.exp(logvar) - logvar - 1.0).sum(dim=-1)


def reparameterize(mu, logvar, eps):
    return mu + torch.exp(0.5 * logvar) * eps


mu = torch.tensor([[1.0, 0.0], [0.5, -0.5]], dtype=torch.float64, requires_grad=True)
logvar = torch.tensor([[0.0, math.log(4.0)], [math.log(0.25), 0.0]], dtype=torch.float64, requires_grad=True)
eps = torch.tensor([[1.0, -1.0], [2.0, 0.0]], dtype=torch.float64)
kl = kl_to_standard_normal(mu, logvar)
z = reparameterize(mu, logvar, eps)
print("kl =", kl)
print("z =", z)
assert abs(float(kl.detach()[0]) - (2 - math.log(2))) < 1e-12                       # Part 3.1: 2 - ln 2
assert torch.allclose(z, torch.tensor([[2.0, -2.0], [1.5, -0.5]], dtype=torch.float64), atol=1e-12, rtol=0)
z.sum().backward()
assert torch.allclose(mu.grad, torch.ones_like(mu), atol=1e-12, rtol=0)
assert torch.allclose(logvar.grad, 0.5 * torch.exp(0.5 * logvar.detach()) * eps, atol=1e-12, rtol=0)
# kl[1] = 1/2 [(1/4 + 1/4 + ln 4 - 1) + (1/4)] = -1/8 + ln 2
assert abs(float(kl.detach()[1]) - (math.log(2) - 0.125)) < 1e-12
print(f"Part 3.5 answer: kl[1] = {float(kl.detach()[1]):.6f}")

In [ ]:
ANSWER = 0.568147
assert abs(ANSWER - float(kl.detach()[1])) <= 1e-6


## Part 3.6 — the β-weighted batch objective

In [ ]:
def beta_vae_loss(x, x_hat, mu, logvar, beta):
    recon_mean = (0.5 * (x - x_hat) ** 2).sum(dim=-1).mean()
    kl_mean = kl_to_standard_normal(mu, logvar).mean()
    total = recon_mean + beta * kl_mean
    return total, recon_mean, kl_mean


f64 = dict(dtype=torch.float64)
xb = torch.tensor([[1.0, -1.0, 2.0], [0.0, 3.0, -2.0]], **f64)
xb_hat = torch.tensor([[0.5, -1.0, 1.0], [0.0, 2.0, -2.0]], **f64)
mub = torch.tensor([[0.5, 0.0], [1.0, -1.0]], **f64)
lvb = torch.tensor([[0.0, 0.0], [0.0, math.log(2.0)]], **f64)
total, recon_mean, kl_mean = beta_vae_loss(xb, xb_hat, mub, lvb, 2.0)
print("total, recon_mean, kl_mean =", float(total), float(recon_mean), float(kl_mean))
assert abs(float(total) - (35 / 16 - 0.5 * math.log(2))) < 1e-12            # Part 3.2
assert abs(float(total) - float(recon_mean + 2 * kl_mean)) < 1e-12
t1, r1, k1 = beta_vae_loss(xb, xb_hat, mub, lvb, 1.0)
assert abs(float(t1) - float(r1 + k1)) < 1e-12
print(f"Part 3.6 answer: total = {float(total):.6f}")

In [ ]:
ANSWER = 1.840926
assert abs(ANSWER - float(total)) <= 1e-6


## Part 3.7 — TinyVAE and its training protocol

In [ ]:
class TinyVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(8, 16)
        self.mu_head = nn.Linear(16, 2)
        self.logvar_head = nn.Linear(16, 2)
        self.dec1 = nn.Linear(2, 16)
        self.dec2 = nn.Linear(16, 8)

    def encode(self, x):
        h = F.relu(self.hidden(x))
        return self.mu_head(h), self.logvar_head(h)

    def decode(self, z):
        return self.dec2(F.relu(self.dec1(z)))

    def forward(self, x, eps):
        mu, logvar = self.encode(x)
        z = reparameterize(mu, logvar, eps)
        return self.decode(z), mu, logvar


def train_vae(beta):
    torch.manual_seed(SEED)
    model = TinyVAE()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    eps_gen = torch.Generator().manual_seed(SEED)
    eval_gen = torch.Generator().manual_seed(SEED + 1)
    eps_heldout = torch.randn(64, 2, generator=eval_gen)
    with torch.no_grad():
        held_before = float(beta_vae_loss(x_heldout, *model(x_heldout, eps_heldout), beta)[0])
    log = []
    for _ in range(400):
        eps = torch.randn(256, 2, generator=eps_gen)
        optimizer.zero_grad(set_to_none=True)
        x_hat, mu, logvar = model(x_train, eps)
        total, recon, kl = beta_vae_loss(x_train, x_hat, mu, logvar, beta)
        total.backward()
        optimizer.step()
        log.append((total.item(), recon.item(), kl.item()))
    with torch.no_grad():
        held_after = float(beta_vae_loss(x_heldout, *model(x_heldout, eps_heldout), beta)[0])
    return model, log, held_before, held_after


model2, log2, held_before2, held_after2 = train_vae(BETA)
totals = [t for t, _, _ in log2]
print(f"first logged total {totals[0]:.3f}; mean of last 20 {sum(totals[-20:]) / 20:.3f}")
print(f"held-out beta-weighted objective (beta = 2), nats per example, constant dropped: "
      f"before {held_before2:.3f}, after {held_after2:.3f}")
assert all(abs(t - (r + 2 * k)) <= 1e-5 for t, r, k in log2)
assert sum(totals[-20:]) / 20 < totals[0]
assert held_after2 <= 0.8 * held_before2

# Gradient-flow probe: the reconstruction term alone must reach the encoder's mean head.
e = torch.randn(256, 2, generator=torch.Generator().manual_seed(SEED + 2))
model2.zero_grad(set_to_none=True)
_, recon_only, _ = beta_vae_loss(x_train, *model2(x_train, e), BETA)
recon_only.backward()
g = model2.mu_head.weight.grad
assert g is not None and bool((g != 0).any())
model2.zero_grad(set_to_none=True)
print(f"Part 3.7 answer: held-out beta-weighted objective (beta = 2) after training = {held_after2:.3f}")

In [ ]:
ANSWER = 2.732
assert abs(ANSWER - held_after2) <= 0.05


## Part 3.8 — active latents and posterior collapse at β = 8

In [ ]:
def per_latent_heldout_kl(model):
    with torch.no_grad():
        mu, logvar = model.encode(x_heldout)
        return (0.5 * (mu ** 2 + torch.exp(logvar) - logvar - 1.0)).mean(dim=0)


kl2 = per_latent_heldout_kl(model2)
active2 = int((kl2 > 0.05).sum())
print("beta = 2 per-latent held-out KL:", [round(v, 4) for v in kl2.tolist()], "active:", active2)

model8, log8, held_before8, held_after8 = train_vae(8.0)
kl8 = per_latent_heldout_kl(model8)
active8 = int((kl8 > 0.05).sum())
print("beta = 8 per-latent held-out KL:", [round(v, 4) for v in kl8.tolist()], "active:", active8)
print(f"beta = 8 last logged kl_mean {log8[-1][2]:.4f} vs beta = 2 {log2[-1][2]:.4f}; "
      f"last recon_mean {log8[-1][1]:.3f} vs {log2[-1][1]:.3f}")

for v in (kl2, kl8):
    assert v.shape == (2,) and bool((v >= 0).all())
assert isinstance(active2, int) and isinstance(active8, int)
print(f"Part 3.8 answer: {active2}, {active8}")

## Answer to Part 3.8(c)

The β = 8 run shows complete posterior collapse: the KL penalty drives the encoder to $q(z\mid x)\approx p(z)$ for both latent coordinates (0 active, against 1 active at β = 2), so $z$ carries no information about $x$ and the decoder outputs roughly the data mean.
The logged `kl_mean` reveals it, falling to about $0.001$ nats while `recon_mean` stays near $3.56$, far above the β = 2 run's $1.21$.

### Answer check

In [ ]:
print(f"3.5: {float(kl.detach()[1]):.6f}")
print(f"3.6: {float(total):.6f}")
print(f"3.7: {held_after2:.3f}")
print(f"3.8: {active2}, {active8}")
assert active8 < active2